# Linear Reduced GP: End-to-End High-Dimensional BO

PCA / PLS / Random Projectionによるsurrogate-side input reductionと、candidate searchの座標系を区別します。このNotebookでは`PCAGP`を代表として実際のcandidate generationまで実行します。

関連理論:

- [Dimensionality Reduction GP](../../docs/theory/18_dimensionality_reduction_gp.md)
- [High-dimensional Search Strategies](../../docs/theory/20_high_dimensional_search.md)

```text
original/public X (10D)
        ↓
    PCAGP reducer
        ↓
model latent Z_m (3D)
        ↓
   GP posterior
        ↓
 acquisition(X)
        ↓
optimize in original/public X (10D)
        ↓
 candidate X (10D)
```

重要なのは、model latent `Z_m`とcandidate-search coordinateを同一視しないことです。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import PCAGP, PLSGP, RandomProjectionGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Sparse high-dimensional objective

10次元入力ですが、目的関数へ強く効くのは一部のdimensionだけにします。

In [ ]:
input_dim = 10
n_train = 28
bounds = torch.stack(
    [torch.zeros(input_dim), torch.ones(input_dim)],
    dim=0,
)


def objective(X: torch.Tensor) -> torch.Tensor:
    value = (
        torch.sin(2 * torch.pi * X[..., 0])
        + 0.6 * (X[..., 3] - 0.3) ** 2
        - 0.5 * X[..., 7]
    )
    return value.unsqueeze(-1)


train_X = torch.rand(n_train, input_dim)
train_Y = objective(train_X)
print(train_X.shape, train_Y.shape)

## 3. Compare linear reducers

すべてのmodelへ渡すtraining inputはoriginal 10Dです。reducerだけがmodel内部の表現を変えます。

In [ ]:
models = {
    "PCA": PCAGP(train_X, train_Y, n_components=3),
    "PLS": PLSGP(train_X, train_Y, n_components=3),
    "Random": RandomProjectionGP(
        train_X,
        train_Y,
        n_components=3,
        random_state=0,
    ),
}

for name, model in models.items():
    print(
        name,
        "public=", model.original_input_dim,
        "latent=", model.reduced_input_dim,
    )

## 4. Fit the representative PCAGP

`PCAGP`はpublic/raw inputを保持したまま内部でPCA表現を使います。

In [ ]:
model = models["PCA"]
fit_gpytorch_mll(model.make_mll())
model.eval()

print("raw_train_X:", model.raw_train_X.shape)
print("original input dim:", model.original_input_dim)
print("reduced input dim:", model.reduced_input_dim)

## 5. Posterior and sampling in public coordinates

callerはlatent coordinateを作らず、original-space Xを`posterior()`へ渡します。

In [ ]:
test_X = torch.rand(4, input_dim)
with torch.no_grad():
    posterior = model.posterior(test_X)
    samples = posterior.rsample(torch.Size([16]))

print("test_X:", test_X.shape)
print("posterior mean:", posterior.mean.shape)
print("samples:", samples.shape)

## 6. Acquisition function

MC acquisitionもoriginal/public Xを受け取ります。内部reductionはmodelの責務です。

In [ ]:
acqf = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
)
probe_X = torch.full((1, 1, input_dim), 0.5)
print("acquisition probe:", acqf(probe_X))

## 7. Candidate generation in original space

PCAが3次元だからといって`optimize_acqf`へ3次元boundsを渡しません。このworkflowではcandidate searchはoriginal 10D boxで行われ、返るcandidateも10Dです。

In [ ]:
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, input_dim])
assert torch.all(candidate >= bounds[0])
assert torch.all(candidate <= bounds[1])
print("candidate shape:", candidate.shape)
print("acquisition value:", acq_value)

## 8. Candidate evaluation and update

In [ ]:
candidate_Y = objective(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate objective:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 9. One refit step

次のiterationではoriginal-space observationを追加してreducer/modelを再構築します。latent coordinateをtraining dataとして保存しません。

In [ ]:
next_model = PCAGP(updated_X, updated_Y, n_components=3)
fit_gpytorch_mll(next_model.make_mll())
next_model.eval()
next_acqf = qLogExpectedImprovement(
    model=next_model,
    best_f=updated_Y.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
)
next_candidate, _ = optimize_acqf(
    acq_function=next_acqf,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)

assert next_candidate.shape[-1] == input_dim
print("next candidate:", next_candidate)

## 10. Surrogate reduction is not search reduction

```text
PCAGP / PLSGP / RandomProjectionGP
    reduce model representation
    public candidate remains original X

REMBO / HeSBO / ALEBO / BAxUS
    change candidate-search geometry
    search target Z_s is a separate coordinate system

TuRBO
    keeps original coordinates but localizes candidate search
```

したがってPCAを使っただけで高次元`optimize_acqf`の難しさまで解消したとは言えません。

## 11. Other high-dimensional surrogate families

- Fully Bayesian SAAS: sparse relevanceをNUTSで推論します。`08_saas_gp.ipynb`で学習contractを扱います。
- MAP-SAAS / additive GP: より軽量なexact-GP系です。`09_map_saas_and_additive_gp.ipynb`で比較します。
- Neural reduction: nonlinear representationを`20_neural_reduction_gp.ipynb`で扱います。
- `23_high_dimensional_bo_benchmark.ipynb`: 複数seedの有限candidate-pool benchmarkでsurrogate差を比較します。

これらを1つの巨大Notebookへ統合せず、ここではpublic-space candidate contractを代表的に実証します。

## 12. When to use and caveats

- PCAはXの分散方向を使うため、目的Yに重要な低分散方向を落とす可能性があります。
- PLSはsupervisedですが、少数データではprojection推定自体が不安定になり得ます。
- Random Projectionはdata-driven reducerではなく、固定されたランダム写像です。
- SAASは「少数dimensionが重要」というsparsity仮定を利用し、input reductionとは別アプローチです。
- 高次元でcandidate optimizationがボトルネックなら、surrogate変更だけでなくTuRBO / REMBO / ALEBO / BAxUS等のsearch strategyを検討します。
- model latent `Z_m`とsearch target `Z_s`を混同しません。

## 13. References

高次元モデルの仮定とsearch strategyの違いは冒頭のTheory章を参照してください。TuRBO / Trust Regionは後続PhaseでEnd-to-End workflowとして扱います。